# Warm up
Let's start with some warm up exercises to get you familiar with the database and do some SQL querying.
Make sure to have downloaded the NBA dataset form Kaggle as described in the README.

In [ ]:
## TODO: Use pathlib to get the path to the data directory. Path(__file__) won't work in a Jupyter notebook.
import sqlite3 as sql
import pandas as pd
from pathlib import Path

# Locate the data directory
DATA_DIR = Path.cwd() / "data"

# Find the SQLite database file
database_files = [
    file for file in DATA_DIR.rglob("*")
    if file.is_file() and file.suffix.lower() in {".sqlite", ".sqlite3", ".db"}
]

# Check that exactly one database file is found
if len(database_files) != 1:
    raise FileNotFoundError(
        f"Expected one SQLite database, but found: {database_files}"
    )

DATABASE_PATH = database_files[0]

print(f"Database location: {DATABASE_PATH}")

# Connect to the database
con = sql.connect(str(DATABASE_PATH))

## Create a SQLite connection and run some SQL queries

In [ ]:
query = """
SELECT *
FROM game
LIMIT 10
"""

top_10_game = pd.read_sql_query(query, con)

top_10_game.head()

In [ ]:
# let's look at the schema of the database
# TODO: sqlite_master is the name of the table you want to inspeact: Replace TABLE_NAME with the name of the table you want to inspect
query = """
SELECT *
FROM sqlite_master
"""

schema = pd.read_sql_query(query, con)

schema.sort_values("name")

In [ ]:
# let's see all the tables we have in the database
# TODO: Filter for tables only
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
"""

tables = pd.read_sql_query(query, con)

tables

In [ ]:
# let's have a look at the columns in the game table
game_columns = pd.read_sql_query(
    "PRAGMA table_info(game)",
    con
)

game_columns

In [ ]:
draft_columns = pd.read_sql_query(
    "PRAGMA table_info(draft_combine_stats)",
    con
)

draft_columns

In [ ]:
# let's find the earliest recorded game
query = """
    SELECT game_date
    FROM game
    ORDER BY game_date
    LIMIT 1
"""

print(pd.read_sql(query, con))

# let's find the latest recorded game
query = """
SELECT game_date
FROM game
ORDER BY game_date DESC
LIMIT 1
"""

print(pd.read_sql(query, con))


In [ ]:
# TODO: Find four interesting facts about the data and write a query to find the answer

## Insight 1：The Evolution of NBA Scoring

In [ ]:
overview_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT game_id) AS unique_games,
    MIN(game_date) AS earliest_game,
    MAX(game_date) AS latest_game,
    SUM(
        CASE
            WHEN pts_home IS NULL OR pts_away IS NULL
            THEN 1 ELSE 0
        END
    ) AS missing_scores
FROM game
"""

overview = pd.read_sql_query(overview_query, con)

display(overview)

In [ ]:
season_type_query = """
SELECT
    season_type,
    COUNT(*) AS number_of_games
FROM game
GROUP BY season_type
ORDER BY number_of_games DESC
"""

season_types = pd.read_sql_query(season_type_query, con)

display(season_types)

In [ ]:
wl_query = """
SELECT
    wl_home,
    COUNT(*) AS number_of_games
FROM game
GROUP BY wl_home
"""

wl_results = pd.read_sql_query(wl_query, con)

display(wl_results)

In [ ]:
season_query = """
SELECT
    season_id,
    MIN(game_date) AS first_game,
    MAX(game_date) AS last_game,
    COUNT(*) AS number_of_games
FROM game
GROUP BY season_id
ORDER BY first_game
"""

seasons = pd.read_sql_query(season_query, con)

display(seasons.head(10))
display(seasons.tail(10))


In [ ]:
duplicate_query = """
WITH duplicate_games AS (
    SELECT
        game_id,
        COUNT(*) AS n_rows
    FROM game
    GROUP BY game_id
    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS duplicated_game_ids,
    SUM(n_rows) AS rows_in_duplicate_groups,
    SUM(n_rows - 1) AS extra_rows
FROM duplicate_games
"""

duplicates = pd.read_sql_query(duplicate_query, con)

display(duplicates)

In [ ]:
duplicate_details_query = """
WITH duplicate_ids AS (
    SELECT game_id
    FROM game
    GROUP BY game_id
    HAVING COUNT(*) > 1
)

SELECT
    g.game_id,
    g.season_id,
    g.season_type,
    g.game_date,
    g.team_name_home,
    g.team_name_away,
    g.pts_home,
    g.pts_away,
    COUNT(*) AS copies
FROM game AS g
INNER JOIN duplicate_ids AS d
    ON g.game_id = d.game_id
GROUP BY
    g.game_id,
    g.season_id,
    g.season_type,
    g.game_date,
    g.team_name_home,
    g.team_name_away,
    g.pts_home,
    g.pts_away
ORDER BY g.game_id
LIMIT 30
"""

duplicate_details = pd.read_sql_query(
    duplicate_details_query, con
)

display(duplicate_details)

In [ ]:
regular_duplicate_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT game_id) AS unique_games,
    COUNT(*) - COUNT(DISTINCT game_id) AS extra_rows
FROM game
WHERE season_type = 'Regular Season'
"""

regular_duplicates = pd.read_sql_query(
    regular_duplicate_query, con
)

display(regular_duplicates)

In [ ]:
scoring_query = """
WITH regular_season_games AS (
    SELECT
        CAST(SUBSTR(CAST(season_id AS TEXT), 2, 4) AS INTEGER)
            AS season_start_year,
        game_id,
        pts_home + pts_away AS total_points

    FROM game

    WHERE season_type = 'Regular Season'
      AND pts_home IS NOT NULL
      AND pts_away IS NOT NULL
)

SELECT
    season_start_year,

    printf(
        '%d-%02d',
        season_start_year,
        (season_start_year + 1) % 100
    ) AS season,

    COUNT(*) AS number_of_games,

    ROUND(AVG(total_points), 2) AS avg_total_points

FROM regular_season_games

GROUP BY season_start_year

ORDER BY season_start_year
"""

scoring_trends = pd.read_sql_query(scoring_query, con)

display(scoring_trends)

In [ ]:
print("Number of seasons:", len(scoring_trends))

print(
    "Total regular-season games:",
    scoring_trends["number_of_games"].sum()
)

display(scoring_trends.head(10))
display(scoring_trends.tail(10))

In [ ]:
highest_scoring = scoring_trends.loc[
    scoring_trends["avg_total_points"].idxmax()
]

lowest_scoring = scoring_trends.loc[
    scoring_trends["avg_total_points"].idxmin()
]

print("Highest-scoring season:")
display(highest_scoring)

print("Lowest-scoring season:")
display(lowest_scoring)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    scoring_trends["season_start_year"],
    scoring_trends["avg_total_points"],
    marker="o",
    markersize=3,
    linewidth=1.8
)

plt.title(
    "Evolution of Average NBA Game Scoring (1946–2023)",
    fontsize=15
)

plt.xlabel("NBA Season", fontsize=12)
plt.ylabel("Average Total Points per Game", fontsize=12)

# Display one tick every five seasons
plt.xticks(
    scoring_trends["season_start_year"].iloc[::5],
    scoring_trends["season"].iloc[::5],
    rotation=45
)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
all_years = set(
    range(
        scoring_trends["season_start_year"].min(),
        scoring_trends["season_start_year"].max() + 1
    )
)

observed_years = set(scoring_trends["season_start_year"])

missing_years = sorted(all_years - observed_years)

print("Missing season start years:", missing_years)

### Findings

The analysis reveals a non-linear evolution of NBA scoring over time.

Average total points per game increased from 135.54 in the 1946-47 season to a historical peak of 233.39 in 1969-70. Scoring subsequently declined, reaching relatively low levels in the late 1990s, before recovering steadily in the 2000s and 2010s.

By the 2022-23 season, average total points had reached 229.37, close to the historical peak.

Overall, average game scoring increased by approximately 69.2% between the first and latest seasons in the dataset. However, the trend was not continuous, highlighting substantial changes in NBA scoring patterns across different eras.

### Conclusion

NBA scoring has undergone distinct historical phases rather than increasing steadily over time. Although modern NBA games are high-scoring, the 1969-70 season recorded the highest average total points in our dataset.

## Insight 2：The Three-Point Revolution

In [ ]:
three_point_quality_query = """
SELECT
    COUNT(*) AS total_games,

    SUM(
        CASE
            WHEN fg3a_home IS NULL
              OR fg3a_away IS NULL
              OR fga_home IS NULL
              OR fga_away IS NULL
            THEN 1
            ELSE 0
        END
    ) AS missing_shooting_stats

FROM game
WHERE season_type = 'Regular Season'
"""

three_point_quality = pd.read_sql_query(
    three_point_quality_query,
    con
)

display(three_point_quality)

In [ ]:
three_point_query = """
WITH regular_season_games AS (

    SELECT
        CAST(
            SUBSTR(CAST(season_id AS TEXT), 2, 4)
            AS INTEGER
        ) AS season_start_year,

        fg3a_home + fg3a_away AS total_3pa,

        fga_home + fga_away AS total_fga

    FROM game

    WHERE season_type = 'Regular Season'

      AND fg3a_home IS NOT NULL
      AND fg3a_away IS NOT NULL
      AND fga_home IS NOT NULL
      AND fga_away IS NOT NULL
)

SELECT

    season_start_year,

    printf(
        '%d-%02d',
        season_start_year,
        (season_start_year + 1) % 100
    ) AS season,

    COUNT(*) AS number_of_games,

    ROUND(
        AVG(total_3pa),
        2
    ) AS avg_3pa_per_game,

    ROUND(
        SUM(total_3pa) * 100.0
        / NULLIF(SUM(total_fga), 0),
        2
    ) AS three_point_attempt_share

FROM regular_season_games

GROUP BY season_start_year

ORDER BY season_start_year
"""

three_point_trends = pd.read_sql_query(
    three_point_query,
    con
)

display(three_point_trends)

In [ ]:
print(
    "Number of seasons:",
    len(three_point_trends)
)

display(three_point_trends.head(10))

display(three_point_trends.tail(10))

In [ ]:
highest_three_point_season = three_point_trends.loc[
    three_point_trends["three_point_attempt_share"].idxmax()
]

display(highest_three_point_season)

In [ ]:
highest_three_point_volume = three_point_trends.loc[
    three_point_trends["avg_3pa_per_game"].idxmax()
]

display(highest_three_point_volume)

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    three_point_trends["season_start_year"],
    three_point_trends["three_point_attempt_share"],
    marker="o",
    markersize=3,
    linewidth=1.8
)

plt.title(
    "The Three-Point Revolution in NBA History",
    fontsize=15
)

plt.xlabel("NBA Season", fontsize=12)

plt.ylabel(
    "Three-Point Attempts (% of All Field Goal Attempts)",
    fontsize=11
)

plt.xticks(
    three_point_trends["season_start_year"].iloc[::5],
    three_point_trends["season"].iloc[::5],
    rotation=45
)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
coverage_query = """
WITH season_data AS (

    SELECT

        CAST(
            SUBSTR(CAST(season_id AS TEXT), 2, 4)
            AS INTEGER
        ) AS season_start_year,

        CASE
            WHEN fg3a_home IS NOT NULL
             AND fg3a_away IS NOT NULL
             AND fga_home IS NOT NULL
             AND fga_away IS NOT NULL
            THEN 1
            ELSE 0
        END AS has_complete_stats

    FROM game

    WHERE season_type = 'Regular Season'
)

SELECT

    season_start_year,

    printf(
        '%d-%02d',
        season_start_year,
        (season_start_year + 1) % 100
    ) AS season,

    COUNT(*) AS total_games,

    SUM(has_complete_stats) AS games_with_stats,

    COUNT(*) - SUM(has_complete_stats) AS missing_games,

    ROUND(
        100.0 * SUM(has_complete_stats) / COUNT(*),
        2
    ) AS coverage_pct

FROM season_data

WHERE season_start_year >= 1979

GROUP BY season_start_year

ORDER BY season_start_year
"""

season_coverage = pd.read_sql_query(
    coverage_query,
    con
)

display(season_coverage)

In [ ]:
three_point_query = """
WITH regular_season_games AS (

    SELECT

        CAST(
            SUBSTR(CAST(season_id AS TEXT), 2, 4)
            AS INTEGER
        ) AS season_start_year,

        fg3a_home + fg3a_away AS total_3pa,

        fga_home + fga_away AS total_fga

    FROM game

    WHERE season_type = 'Regular Season'

      AND fg3a_home IS NOT NULL
      AND fg3a_away IS NOT NULL
      AND fga_home IS NOT NULL
      AND fga_away IS NOT NULL

      AND CAST(
          SUBSTR(CAST(season_id AS TEXT), 2, 4)
          AS INTEGER
      ) >= 1985
)

SELECT

    season_start_year,

    printf(
        '%d-%02d',
        season_start_year,
        (season_start_year + 1) % 100
    ) AS season,

    COUNT(*) AS number_of_games,

    ROUND(
        AVG(total_3pa),
        2
    ) AS avg_3pa_per_game,

    ROUND(
        SUM(total_3pa) * 100.0
        / NULLIF(SUM(total_fga), 0),
        2
    ) AS three_point_attempt_share

FROM regular_season_games

GROUP BY season_start_year

ORDER BY season_start_year
"""

three_point_trends = pd.read_sql_query(
    three_point_query,
    con
)

display(three_point_trends)

In [ ]:
missing_season_query = """
SELECT
    season_id,
    season_type,
    COUNT(*) AS number_of_games,
    MIN(game_date) AS first_game,
    MAX(game_date) AS last_game

FROM game

WHERE CAST(
    SUBSTR(CAST(season_id AS TEXT), 2, 4)
    AS INTEGER
) BETWEEN 2011 AND 2013

GROUP BY season_id, season_type

ORDER BY season_id, season_type
"""

missing_season_check = pd.read_sql_query(
    missing_season_query,
    con
)

display(missing_season_check)

In [ ]:
plot_data = (
    three_point_trends
    .set_index("season_start_year")
    .reindex(range(1985, 2023))
)

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    plot_data.index,
    plot_data["three_point_attempt_share"],
    marker="o",
    markersize=4,
    linewidth=2
)

plt.title(
    "The Three-Point Revolution in NBA History (1985–2023)",
    fontsize=15
)

plt.xlabel("NBA Season", fontsize=12)

plt.ylabel(
    "Three-Point Attempts (% of All Field Goal Attempts)",
    fontsize=11
)

# Display one tick every five years
tick_years = list(range(1985, 2023, 5))

plt.xticks(
    tick_years,
    [
        f"{year}-{(year + 1) % 100:02d}"
        for year in tick_years
    ],
    rotation=45
)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
initial_season = three_point_trends.iloc[0]

latest_season = three_point_trends.iloc[-1]

highest_share_season = three_point_trends.loc[
    three_point_trends["three_point_attempt_share"].idxmax()
]

highest_volume_season = three_point_trends.loc[
    three_point_trends["avg_3pa_per_game"].idxmax()
]

print("Initial season:")
display(initial_season)

print("Latest season:")
display(latest_season)

print("Highest three-point attempt share:")
display(highest_share_season)

print("Highest average three-point attempts:")
display(highest_volume_season)

In [ ]:
share_change = (
    latest_season["three_point_attempt_share"]
    - initial_season["three_point_attempt_share"]
)

print(
    f"Change in three-point attempt share: "
    f"{share_change:.2f} percentage points"
)

## Insight 2: The Three-Point Revolution

### Research Question

How has the importance of three-point shooting evolved in the NBA?

### Methodology

We analyse regular-season games from 1985-86 to 2022-23, using seasons with complete shooting statistics. Two indicators are calculated: average three-point attempts per game and three-point attempts as a percentage of all field goal attempts.

The percentage is calculated using aggregate three-point attempts divided by aggregate field goal attempts.

The 2012-13 season is absent from the source database and is therefore shown as a gap in the time series.

### Findings

The results reveal a substantial increase in the importance of three-point shooting.

In 1985-86, three-point attempts accounted for only 3.76% of all field goal attempts. This share increased to a peak of 39.93% in 2021-22, representing an increase of 36.17 percentage points.

Average three-point attempts per game also increased from 6.66 to 68.43 between 1985-86 and 2022-23.

Although the three-point attempt share declined slightly to 38.75% in 2022-23, it remained close to its historical peak.

### Conclusion

Three-point shooting has become a central component of modern NBA offensive strategies. The historical trend demonstrates a major transformation in shot selection, with teams increasingly relying on three-point attempts.